# Advanced Google Search Tool in Python

An upgraded version of the original notebook. It supports single and multiple searches, Google search operators, safe query validation, optional opening of multiple tabs, and a reusable function design.

**How it works:** it builds a Google Search URL and opens it in your default browser. It does not scrape Google results or require an API key.

**Requirements:** Python 3 (uses only the Python standard library).

In [ ]:
from urllib.parse import quote_plus
from webbrowser import open as open_browser
from time import sleep
from typing import Iterable

MAX_QUERY_LENGTH = 300
MAX_BATCH_QUERIES = 10


def build_google_url(query: str) -> str:
    """Build a Google Search URL from a non-empty query."""
    if not isinstance(query, str):
        raise TypeError("Search query must be a string.")

    query = query.strip()
    if not query:
        raise ValueError("Search query cannot be empty.")
    if len(query) > MAX_QUERY_LENGTH:
        raise ValueError(f"Search query must be {MAX_QUERY_LENGTH} characters or fewer.")

    return f"https://www.google.com/search?q={quote_plus(query)}"


def search_google(query: str, open_tab: bool = True) -> str:
    """Display the search URL and optionally open it in the default browser."""
    url = build_google_url(query)
    print(f"Searching Google for: {query.strip()}")
    print(f"URL: {url}")
    if open_tab:
        opened = open_browser(url, new=2)  # new tab if possible
        if not opened:
            print("The browser could not be opened automatically. Copy the URL above.")
    return url


def search_many(queries: Iterable[str], open_tabs: bool = True, delay_seconds: float = 1.0) -> list[str]:
    """Search up to 10 queries. A short delay avoids opening tabs all at once."""
    cleaned_queries = [q.strip() for q in queries if isinstance(q, str) and q.strip()]
    if not cleaned_queries:
        raise ValueError("Provide at least one non-empty search query.")
    if len(cleaned_queries) > MAX_BATCH_QUERIES:
        raise ValueError(f"A maximum of {MAX_BATCH_QUERIES} searches is allowed per batch.")
    if delay_seconds < 0 or delay_seconds > 10:
        raise ValueError("delay_seconds must be between 0 and 10.")

    urls = []
    for index, query in enumerate(cleaned_queries):
        urls.append(search_google(query, open_tab=open_tabs))
        if open_tabs and delay_seconds and index < len(cleaned_queries) - 1:
            sleep(delay_seconds)
    return urls


print("Google Search Tool is ready.")

## 1. Run a single search

Change the query to anything you want to search for. Set `open_tab=False` if you only want to generate the URL.

In [ ]:
search_google("Python programming tutorials")

## 2. Search multiple topics

This example opens one browser tab per query. Set `open_tabs=False` to print URLs only.

In [ ]:
queries = [
    "Python list comprehension examples",
    "site:docs.python.org pathlib",
    '"data analyst" SQL Power BI jobs',
]

search_many(queries, open_tabs=True, delay_seconds=1)

## 3. Use Google search operators

Useful operators include:
- `site:python.org tutorial` — search within a website
- `"exact phrase"` — search for an exact phrase
- `filetype:pdf machine learning` — find PDF files
- `python OR java tutorial` — search either term
- `-jobs Python` — exclude a term

These are examples; Google controls how operators are interpreted.

In [ ]:
search_google('filetype:pdf machine learning syllabus', open_tab=False)

## 4. Interactive search

Run this cell and enter a query when prompted. In some notebook environments, browser opening may be blocked; the URL will still be printed.

In [ ]:
query = input("Enter your Google search query: ").strip()
if query:
    search_google(query)
else:
    print("No query entered. Please run the cell again and enter a search term.")

## Improvements over the basic version

- Reusable functions instead of repeated code
- URL encoding for spaces, quotes, and special characters
- Empty-query and length validation
- Batch search with a limit and optional delay
- Option to generate URLs without opening a browser
- Uses the Python standard library, so `pywhatkit` is not required

**Privacy note:** search terms are sent to Google when you open the generated URLs. Avoid putting passwords or confidential information into a search query.